<div style="background: linear-gradient(135deg, #0A2540, #1E40AF); padding: 40px; border-radius: 15px; color: white; text-align: center;">
<div style="text-align:center;">
<h1 style="font-size: 42px;"> Case Técnico</h1>

<h2 style="font-size: 28px; margin-top: 10px;">
Previsão de Casos de Dengue com Machine Learning
</h2>

<br>

<p style="font-size: 18px;">
 <b>Nome da candidata:</b> Micaeli M. Theodoro <br>
 <b>Vaga:</b> Data Science / A.I.
</p>

</div>

<div style="border-left: 6px solid #2563EB; padding: 20px; background-color: #F1F5F9; border-radius: 10px;">

<h2 style="color:#1E3A8A;"> Problema</h2>

<p style="font-size:16px;">
A Dengue apresenta forte dependência temporal e influência de fatores climáticos, tornando a previsão um problema não trivial.
</p>

<ul>
<li>Dependência temporal</li>
<li>Variáveis exógenas (temperatura e precipitação)</li>
<li>Relações não lineares
</ul>
<p><b>Objetivo:</b> Prever casos ativos de Dengue ao longo do tempo.</p>

<div style="text-align:center;">
    <h3 style="color:#1E3A8A;">
        Ciclo de vida do vetor depende altamente de condições climáticas
    </h3>
    <img src="ciclo_vida_mosquito.png" width="500"/>
</div>
</div>

<div style="background-color: #EFF6FF; padding: 20px; border-radius: 10px;">

<h2 style="color:#1D4ED8;">Feature Engineering</h2>

<ul>
<li>Transformação de série temporal em formato tabular</li>
<li>Criação de lags: lag 1-7</li>
<li>Médias móveis de casos: 3 e 5 semanas</li>
<li>Sazonalidade - Funções trigonométricas</li>
<li>Clima defasado</li>
</ul>

<p>
Essas técnicas permitem capturar padrões temporais e relações não lineares entre variáveis.
</p>

</div>

In [ ]:
# Lags
for i in range(1, 8):
    df[f'lag_{i}'] = df['ativos'].shift(i)

# Médias móveis de 3 e 5 semanas
df['media_3'] = df['ativos'].rolling(3).mean()
df['media_5'] = df['ativos'].rolling(5).mean()

# Sazonalidade utilizando funções trigonométricas de seno e cosseno
df['sin_semana'] = np.sin(2 * np.pi * df['semana'] / 52)
df['cos_semana'] = np.cos(2 * np.pi * df['semana'] / 52)

# Efeito defasado do clima na dinâmica
for i in range(1, 5):
    df[f'chuva_lag_{i}'] = df['Precipitação'].shift(i)
    df[f'temp_lag_{i}'] = df['Temperatura Média'].shift(i)


<div style="background: linear-gradient(135deg, #DBEAFE, #BFDBFE); padding: 25px; border-radius: 12px;">

<h2 style="color:#1E40AF;">Modelo: Gradient Boosting</h2>

<p>
Modelo baseado em <b>ensembles sequenciais</b>, onde cada árvore aprende corrigindo os erros da anterior.
</p>

<ul>
<li>Captura não linearidade</li>
<li>Modela interações complexas</li>
<li>Bom equilíbrio viés-variância</li>
<li>Robusto com poucos dados</li>
</ul>

<p>
O modelo aprende uma função complexa que relaciona histórico, clima e sazonalidade com os casos de dengue.
</p>

</div>

In [ ]:
# Seleciona todas as colunas exceto a variável alvo ('ativos') e o índice temporal ('semana')
features = [col for col in df.columns if col not in ['ativos', 'semana']]

# X = variáveis explicativas (features)
X = df[features]

# y = variável alvo (casos ativos de dengue)
y = df['ativos']

# Divisão respeitando a ordem temporal (evita vazamento de informação)
split = len(dengue_2023) - 10

# Dados de treino (passado)
X_train = X.iloc[:split]
y_train = y.iloc[:split]

# Dados de teste (futuro)
X_test = X.iloc[split:]
y_test = y.iloc[split:]


# GRADIENT BOOSTING

model = GradientBoostingRegressor(
    n_estimators=300,   # número de árvores (iterações do boosting)
    learning_rate=0.03, # controla o quanto cada árvore corrige o erro
    max_depth=3         # profundidade das árvores (controle de complexidade)
)

# Treinamento do modelo
model.fit(X_train, y_train)


# Resultado da previsão para os dados de teste
y_pred = model.predict(X_test)

<h2 style="color:#1E3A8A;"> Resultado do Modelo</h2>

<p style="font-size:16px;">
Comparação entre valores reais e previstos ao longo do tempo.
</p>

<div style="text-align:center;">
    <h3 style="color:#1E3A8A;">
        Resultado da aplicação de uma metodologia de Machine Learning nos dados de Dengue na cidade de Bauru/SP
    </h3>
    <img src="casosdengue2324.png" width="500"/>
</div>

<div style="background-color:#F8FAFC; padding:20px; border-radius:10px;">

<h2 style="color:#1E3A8A;">Importância das Variáveis</h2>

<p>
Identifica quais variáveis mais contribuíram para a redução do erro do modelo.
</p>

<ul>
<li>Lags → memória da doença</li>
<li>Clima → fatores ambientais</li>
<li>Médias móveis → tendência</li>
</ul>
<div style="text-align:center;">
    <h3 style="color:#1E3A8A;">
        Importância das variáveis utilizadas no modelo de Machine Learning em Dengue 
    </h3>
    <img src="importancia.png" width="500"/>

</div>

<div style="background: linear-gradient(135deg, #E0F2FE, #BAE6FD); padding: 25px; border-radius: 12px;">

<h2 style="color:#075985;">Análise de Sensibilidade (PRCC)</h2>

<p>
O PRCC avalia a influência de cada variável controlando as demais.
</p>

<p>
<b>Diferença chave:</b><br>
Feature Importance → o que o modelo usa<br>
PRCC → o que realmente influencia o sistema
</p>

<p>
Essa análise adiciona profundidade interpretativa ao modelo.
</p>
<div style="text-align:center;">
    <h3 style="color:#1E3A8A;">
        Análise de Sensibilidade (AS) do modelo utilizando o PRCC (Partial Rank Correlation Coefficient)
    </h3>
    <img src="asmachinelearning.png" width="500"/>

</div>
</div>

<div style="border: 2px solid #3B82F6; padding: 20px; border-radius: 12px;">

<h2 style="color:#1D4ED8;">Contribuições</h2>

<ul>
<li>Modelagem do problema</li>
<li>Feature engineering temporal</li>
<li>Treinamento do modelo</li>
<li>Análise interpretativa</li>
</ul>

<p>
<b>Impacto:</b> Modelo com potencial aplicação em sistemas de alerta precoce.
</p>

</div>

<div style="background: linear-gradient(135deg, #1E3A8A, #2563EB); padding: 30px; border-radius: 15px; color: white;">

<h2> Conclusão</h2>

<ul>
<li>Captura de padrões temporais e climáticos</li>
<li>Boa capacidade preditiva</li>
<li>Equilíbrio entre performance e interpretabilidade</li>
</ul>

<p style="margin-top:15px;">
💡 <b>A escolha do modelo foi guiada pela estrutura do problema, não apenas pela performance.</b>
</p>

</div>